# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
md = ds.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references are by `@id`.

Let's enumerate all record sets, including their fields and example columns.

In [ ]:
print("Record Sets:\n")
for rs in ds.metadata.record_sets:
    print(f"@id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '-')}")
    print("  Fields and columns:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}, Name: {field.name}, Data type: {getattr(field, 'data_type', '-')}")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"      Column @id: {col.id} Name: {getattr(col, 'name', '-')}")
    print()
# For example purposes, store the first record set's id
record_set_ids = [rs.id for rs in ds.metadata.record_sets]
print(f"Record set ids found: {record_set_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We will demonstrate with the first record set (replace `record_set_id` with another from above if desired).

In [ ]:
# Extract data from each record set
dataframes = {}
# Use all discovered record set ids
for record_set_id in record_set_ids:
    try:
        records = list(ds.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set: {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Display columns and first rows for the first record set
main_rs_id = record_set_ids[0]
print(f"Columns for record set {main_rs_id}:")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will select a numeric field for filtering and normalization, and group by a key attribute.

Replace the `numeric_field_id` and `group_field_id` with those observed for the dataset.

In [ ]:
# Guess a numeric field (you may adjust this based on printed columns above)
# For this dataset, lets try common protein mass spec columns: 'MW', 'Coverage', 'PeptideCount'
df = dataframes[main_rs_id]
possible_numeric = ['MW', 'MolecularWeight', 'Coverage', 'PeptideCount', 'Abundance']
numeric_field = None
for col in df.columns:
    if any(key.lower() in col.lower() for key in possible_numeric):
        numeric_field = col
        break
if numeric_field is None:
    # fallback to the first numeric column
    numeric_cols = df.select_dtypes(include=np.number).columns
    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]
    else:
        raise RuntimeError('No numeric fields found for EDA.')
print(f"Using numeric field: {numeric_field}")

# Filter: keep values above the median
threshold = df[numeric_field].median()
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Guess a grouping field: typical for protein data might be 'Gene', 'Category', 'Sample', etc
candidate_group_cols = [c for c in df.columns if c.lower() in ['sample', 'condition', 'group', 'category']]
group_field = candidate_group_cols[0] if candidate_group_cols else None

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field} (showing up to 5 rows):")
    print(grouped_df.head())
else:
    print("No grouping field found in columns.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and a scatterplot if groupings exist.

In [ ]:
# Histogram of the numeric field
plt.figure(figsize=(8, 4))
df[numeric_field].hist(bins=40)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If a group field exists, plot boxplot
if group_field and group_field in df.columns:
    plt.figure(figsize=(10,5))
    df.boxplot(column=numeric_field, by=group_field, rot=45)
    plt.title(f'{numeric_field} by {group_field}')
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()
# Scatterplot of two top numeric fields (if more than one available)
numeric_cols = df.select_dtypes(include=np.number).columns
if len(numeric_cols) > 1:
    plt.figure(figsize=(6,6))
    plt.scatter(df[numeric_cols[0]], df[numeric_cols[1]], alpha=0.6)
    plt.xlabel(numeric_cols[0])
    plt.ylabel(numeric_cols[1])
    plt.title(f"{numeric_cols[0]} vs. {numeric_cols[1]}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² mass spectrometry dataset using the `mlcroissant` library following the Croissant schema at its source URL. We:
- Inspected the schema structure, including record sets and fields (referenced by their `@id`s).
- Loaded tabular data dynamically from the available record set(s).
- Demonstrated EDA with filtering, normalization, grouping, and visualizations of key numeric fields.

**Next steps:** You can further query the data by changing `numeric_field`, applying advanced filters, or linking to external protein databases for domain-informed feature engineering.